## Proyecto para experimentos en WandB

In [ ]:
PROJECT = "mnist-mlp"
ENTITY = None

run = wandb.init(
    project=PROJECT, entity=ENTITY, name="exp0_baseline_sigmoid_bce_sgd_mom",
    config={
        "arch": "784-30-10 (sigmoid,sigmoid)",
        "loss": "binary_crossentropy",
        "optimizer": "SGD",
        "lr": learning_rate,
        "momentum": mu,
        "batch_size": batch_size,
        "epochs": epochs,
        "dataset": "MNIST flatten [0,1]"
    },
    reinit=True
)

from wandb.integration.keras import WandbMetricsLogger

history = model.fit(
    x_trainv, y_trainc,
    batch_size=batch_size, epochs=epochs, shuffle=True, verbose=1,
    validation_data=(x_testv, y_testc),
    callbacks=[WandbMetricsLogger(log_freq="epoch")]  # ← sin logging por batch
)

#Proyecto base
#Accuracy final
pred = model.predict(x_testv, batch_size=1024, verbose=0).argmax(1)
final_acc = float((pred == y_test).mean())
wandb.log({"final_argmax_acc": final_acc})
print(f"Baseline Test (argmax): {final_acc*100:.2f}%")
print("Panel W&B:", wandb.run.url)

wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 2:08:21 1s/step - categorical_accuracy: 0.2000 - loss: 0.60 ━━━━━━━━━━━━━━━━━━━━ 52s 9ms/step - categorical_accuracy: 0.1326 - loss: 0.5970   ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - categorical_accuracy: 0.1121 - loss: 0.562 ━━━━━━━━━━━━━━━━━━━━ 35s 6ms/step - categorical_accuracy: 0.1075 - loss: 0.527 ━━━━━━━━━━━━━━━━━━━━ 39s 7ms/step - categorical_accuracy: 0.1065 - loss: 0.515 ━━━━━━━━━━━━━━━━━━━━ 46s 8ms/step - categorical_accuracy: 0.1067 - loss: 0.510 ━━━━━━━━━━━━━━━━━━━━ 50s 9ms/step - categorical_accuracy: 0.1071 - loss: 0.501 ━━━━━━━━━━━━━━━━━━━━ 48s 8ms/step - categorical_accuracy: 0.1082 - loss: 0.486 ━━━━━━━━━━━━━━━━━━━━ 44s 7ms/step - categorical_accuracy: 0.1097 - loss: 0.470 ━━━━━━━━━━━━━━━━━━━━ 41s 7ms/step - categorical_accuracy: 0.1104 - loss: 0.457 ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - categorical_accuracy: 0.1112 - loss: 0.447 ━━━━━━━━━━━━━━━━━━━━ 37s 6ms/step - categorical_accuracy: 0.1149 - loss: 0.436 ━━━━━━━━━━━━━━━━━━━━ 36s 6

In [15]:
baseline_argmax = final_acc
baseline_val_acc = float(history.history['val_categorical_accuracy'][-1])
print(f"Baseline: test(argmax)={baseline_argmax*100:.2f}% | val_acc={baseline_val_acc*100:.2f}%")

Baseline: test(argmax)=95.61% | val_acc=95.61%


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from wandb.integration.keras import WandbMetricsLogger

#Definimos un constructor de modelos
def build_model(cfg):
    model = Sequential([keras.Input(shape=(784,))])
    
    for units in cfg["hidden_layers"]:
        model.add(Dense(units,
                        activation=cfg["activation"],
                        kernel_initializer=cfg.get("kernel_init", "glorot_uniform"),
                        bias_initializer="zeros"))
    if cfg["output_activation"] == "softmax":
        model.add(Dense(10, activation="softmax"))
        loss = "categorical_crossentropy"
    else:
        model.add(Dense(10, activation="sigmoid"))
        loss = "binary_crossentropy"

    opt = cfg["optimizer"].lower(); lr = cfg["lr"]
    if opt == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=lr,
                                         momentum=cfg.get("momentum", 0.0),
                                         nesterov=cfg.get("nesterov", False))
    elif opt == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=lr)
    elif opt == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError("Optimizador no soportado")

    model.compile(optimizer=optimizer, loss=loss, metrics=["categorical_accuracy"])
    return model

#Definimos función para correr experimentos
def run_experiment(name, cfg, epochs=epochs, batch_size=batch_size):
    tf.keras.backend.clear_session()
    run = wandb.init(project=PROJECT, entity=ENTITY, name=name, config=cfg, reinit=True)

    model = build_model(cfg)
    hist = model.fit(
        x_trainv, y_trainc,
        validation_data=(x_testv, y_testc),
        epochs=epochs, batch_size=batch_size, shuffle=True, verbose=1,
        callbacks=[WandbMetricsLogger(log_freq="epoch")]
    )

    #Accuracy final 
    preds = model.predict(x_testv, batch_size=1024, verbose=0).argmax(1)
    final_acc = float((preds == y_test).mean())
    wandb.log({"final_argmax_acc": final_acc})
    print(f"[{name}] test(argmax)={final_acc*100:.2f}%")
    print("Panel W&B:", wandb.run.url)

    wandb.finish()
    return hist, final_acc

experiments = [
    #EXP 1: dos capas + ReLU + SGD Nesterov + softmax
    {
        "name": "exp1_relu_sgd_nesterov",
        "hidden_layers": [64, 32],
        "activation": "relu",
        "output_activation": "softmax",
        "optimizer": "sgd",
        "lr": 0.05,
        "momentum": 0.9,
        "nesterov": True,
        "kernel_init": "he_normal"
    },
    #EXP 2: una capa 128 + ReLU + Adam + softmax
    {
        "name": "exp2_relu_adam",
        "hidden_layers": [128],
        "activation": "relu",
        "output_activation": "softmax",
        "optimizer": "adam",
        "lr": 1e-3,
        "kernel_init": "he_normal"
    },
    #EXP 3: dos capas tanh + RMSprop + softmax
    {
        "name": "exp3_tanh_rmsprop",
        "hidden_layers": [256, 128],
        "activation": "tanh",
        "output_activation": "softmax",
        "optimizer": "rmsprop",
        "lr": 1e-3
    }
]

#Ejecutar bucle
res = []
for cfg in experiments:
    h, acc = run_experiment(cfg["name"], cfg)
    res.append((cfg["name"], acc))

print("\nResumen (test argmax):")
for name, acc in res:
    print(f" - {name}: {acc*100:.2f}%  (Δ vs exp0 = {(acc - baseline_argmax)*100:.2f} pts)")

Epoch 1/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 2:14:30 1s/step - categorical_accuracy: 0.2000 - loss: 2.26 ━━━━━━━━━━━━━━━━━━━━ 2:32 25ms/step - categorical_accuracy: 0.1444 - loss: 2.2762 ━━━━━━━━━━━━━━━━━━━━ 1:41 17ms/step - categorical_accuracy: 0.1545 - loss: 2.279 ━━━━━━━━━━━━━━━━━━━━ 1:41 17ms/step - categorical_accuracy: 0.1856 - loss: 2.242 ━━━━━━━━━━━━━━━━━━━━ 1:11 12ms/step - categorical_accuracy: 0.2565 - loss: 2.117 ━━━━━━━━━━━━━━━━━━━━ 48s 8ms/step - categorical_accuracy: 0.3376 - loss: 1.9175  ━━━━━━━━━━━━━━━━━━━━ 46s 8ms/step - categorical_accuracy: 0.3609 - loss: 1.853 ━━━━━━━━━━━━━━━━━━━━ 38s 6ms/step - categorical_accuracy: 0.4045 - loss: 1.722 ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - categorical_accuracy: 0.4158 - loss: 1.689 ━━━━━━━━━━━━━━━━━━━━ 39s 7ms/step - categorical_accuracy: 0.4344 - loss: 1.636 ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - categorical_accuracy: 0.4471 - loss: 1.601 ━━━━━━━━━━━━━━━━━━━━ 40s 7ms/step - categorical_accuracy: 0.4619 - loss: 1.561 ━━━━━━━━━━━━━━━━━━